# 02a · 무기억성과 경쟁하는 시계 (Memorylessness and Competing Exponential Clocks)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Ross 12e §5.2 (5.2.1 정의, 5.2.2 무기억성, 5.2.3 최솟값·경주·Erlang) · Durrett 3e §2.1 (짧고 증명 중심) · G&S 4e §6.8, §4.4 (엄밀한 서술이 필요할 때만) |
| 선수 노트북 | 00b, 00c |
| 예상 소요 | 75분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **지수 시계는 늙지 않는다.** 위험률이 상수이므로 '얼마나 기다렸는가'는 정보가 아니고, 시계 여러 개를 동시에 돌리면 '언제 울리나'와 '누가 울리나'가 독립으로 갈라진다. 이 그림이 그대로 푸아송 과정(02b)과 CTMC(03a)의 점프 메커니즘이 된다.

## 0. 준비 (Setup)

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy import stats
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.plots import plot_hist_vs_pdf

SEED, rng = rng_for("02a")
setup_plots()
N = scale(200_000)        # 표본 수. FAST(SPKIT_FAST=1) 모드에서는 1/10 크기 (20_000)
print(f"SEED={SEED}, fast={fast()}, N={N}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (01g) 골턴-왓슨 과정의 소멸확률 $q$는 어떤 방정식의 어떤 해인가?

<details><summary>정답</summary>

자손 pgf $\varphi(s)=\sum_k p_k s^k$의 고정점 방정식 $s=\varphi(s)$의 $[0,1]$ 안 **가장 작은** 해. 반복 $q_{n+1}=\varphi(q_n)$, $q_0=0$이 그 해로 단조 수렴한다.

</details>

**Q2.** (01g) 평균 자손 수 $m=\varphi'(1)$이 $m\le 1$이면 소멸확률은? ($p_1<1$ 가정)

<details><summary>정답</summary>

$q=1$ (임계 $m=1$도 포함). $m>1$이면 $q<1$. 이유: $\varphi$가 볼록이고 $\varphi(1)=1$이므로 $\varphi'(1)\le1$이면 $[0,1)$에 다른 고정점이 없다.

</details>

**Q3.** (01g) $Z_n/m^n$이 마팅게일인 이유를 한 줄로.

<details><summary>정답</summary>

$E[Z_{n+1}\mid Z_n]=mZ_n$이므로 $E[Z_{n+1}/m^{n+1}\mid \mathcal F_n]=Z_n/m^n$.

</details>

**Q4.** (01f) 세부 균형 $\pi(x)P(x,y)=\pi(y)P(y,x)$가 모든 $x,y$에서 성립하면 $\pi$는 무엇인가? 역도 성립하나?

<details><summary>정답</summary>

정상분포다: 양변을 $x$에 대해 합하면 $\sum_x\pi(x)P(x,y)=\pi(y)$. 역은 성립하지 않는다 (정상분포는 있지만 가역이 아닌 사슬, 예: 순환 $0\to1\to2\to0$).

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **지수분포** $X\sim\mathrm{Exp}(\lambda)$, $\lambda>0$: 밀도 $f(x)=\lambda e^{-\lambda x}$ ($x\ge0$), 생존함수 $P(X>t)=e^{-\lambda t}$, $E[X]=1/\lambda$, $\mathrm{Var}(X)=1/\lambda^2$. $\lambda$는 **비율(rate)**, $1/\lambda$는 평균. numpy: `rng.exponential(scale=1/lam, size=...)` — scale 인자는 **평균**이다.
- **무기억성(memorylessness)**: 양의 확률변수 $X$가 모든 $s,t\ge0$에 대해 $P(X>s+t\mid X>s)=P(X>t)$를 만족하면 무기억이라 한다. 동치: $P(X>s+t)=P(X>s)P(X>t)$. "이미 $s$만큼 기다렸다"는 정보가 남은 대기시간의 분포를 바꾸지 않는다.
- **위험률(hazard rate)** $h(t)=f(t)/P(X>t)$: 짧은 구간 $(t,t+dt]$에서 '지금까지 안 울린 시계가 울릴' 조건부 비율, $P(X\le t+dt\mid X>t)\approx h(t)\,dt$. $\mathrm{Exp}(\lambda)$는 $h(t)\equiv\lambda$ (상수 위험률) — 무기억성의 미분 버전. 일반적으로 $P(X>t)=\exp\!\big(-\int_0^t h\big)$.
- **경쟁하는 시계(competing clocks)**: 독립 $X_i\sim\mathrm{Exp}(\lambda_i)$, $i=1,\dots,n$. $T=\min_i X_i$ (첫 울림 시각), $I=\arg\min_i X_i$ (누가 울렸나). 독립 연속분포이므로 동률의 확률은 0이다.
- **Erlang / 감마분포**: iid $\mathrm{Exp}(\lambda)$ $n$개의 합 $S_n=X_1+\cdots+X_n\sim\mathrm{Gamma}(n,\lambda)$, 밀도 $f_{S_n}(t)=\lambda e^{-\lambda t}\dfrac{(\lambda t)^{n-1}}{(n-1)!}$, $E[S_n]=n/\lambda$, $\mathrm{Var}(S_n)=n/\lambda^2$. scipy: `stats.gamma(a=n, scale=1/lam)`.

### 2.2 정리 1 — 무기억성의 특성화 (Ross 12e §5.2.2, G&S 4e §4.4)

> 양의 연속 확률변수 $X$가 무기억이면, 즉 모든 $s,t\ge0$에 대해 $P(X>s+t)=P(X>s)P(X>t)$이면, 어떤 $\lambda>0$에 대해 $X\sim\mathrm{Exp}(\lambda)$이다. 역으로 $\mathrm{Exp}(\lambda)$는 무기억이다.

**가정이 왜 필요한가**
- '모든 $s,t$'가 필요하다: 특정 $s$ 하나에서만 $P(X>s+t)=P(X>s)P(X>t)$가 성립하는 분포는 얼마든지 만들 수 있다 (밀도를 조각별로 조정).
- '모든 **실수** $s,t$'가 필요하다: 정수(또는 유리수) $s,t$에서만 요구하면 기하분포도 해가 된다. 함수방정식 $g(s+t)=g(s)g(t)$의 해를 유리수에서 실수로 확장할 때 쓰는 것은 생존함수 $g(t)=P(X>t)$의 단조성뿐이며, 이는 어떤 확률변수에서도 자동으로 성립한다 — 정리의 '연속' 가정은 결론(밀도가 있는 지수분포)을 미리 말해 주는 역할이다.
- 어떤 $t>0$에서 $P(X>t)>0$: 아니면 $X\equiv0$인 퇴화 해가 된다.

<details><summary>증명</summary>

$g(t)=P(X>t)$로 두자. 가정에서 $g(s+t)=g(s)g(t)$ (모든 $s,t\ge0$), $g$는 비증가, 우연속이고 $g(0)=1$이다.

**(i) 유리수에서.** $g(1)=g(\tfrac1n+\cdots+\tfrac1n)=g(\tfrac1n)^n$이므로 $g(\tfrac1n)=g(1)^{1/n}$. 다시 $g(\tfrac mn)=g(\tfrac1n)^m=g(1)^{m/n}$. 즉 모든 양의 유리수 $r$에서 $g(r)=g(1)^r$.

**(ii) $g(1)\in(0,1)$.** $g(1)=0$이면 $g(\tfrac1n)=0$이 모든 $n$에서 성립하고 우연속성으로 $g(0)=\lim_n g(\tfrac1n)=0\ne1$, 모순. $g(1)=1$이면 $g(m)=g(1)^m=1$이 모든 $m$에서 성립해 $P(X>m)=1$ $\forall m$, 즉 $X=\infty$ a.s.가 되어 $X$가 실수값이라는 데 모순. 따라서 $\lambda:=-\log g(1)\in(0,\infty)$이고 유리수 $r$에서 $g(r)=e^{-\lambda r}$.

**(iii) 실수로 확장.** 임의의 $t>0$과 $r_1\le t\le r_2$인 유리수 $r_1,r_2$에 대해 $g$가 비증가이므로 $e^{-\lambda r_2}=g(r_2)\le g(t)\le g(r_1)=e^{-\lambda r_1}$. $r_1\uparrow t$, $r_2\downarrow t$로 보내면 양끝이 $e^{-\lambda t}$로 수렴하므로 $g(t)=e^{-\lambda t}$. 즉 $X\sim\mathrm{Exp}(\lambda)$. (연속성 가정은 여기서 단조성으로만 쓰였다.)

**역방향.** $P(X>s+t)=e^{-\lambda(s+t)}=e^{-\lambda s}e^{-\lambda t}=P(X>s)P(X>t)$. $\square$

</details>

### 2.3 정리 2 — 최솟값과 승자: 경쟁하는 시계 (Durrett 3e §2.1, Ross 12e §5.2.3)

> 독립 $X_i\sim\mathrm{Exp}(\lambda_i)$, $\Lambda=\sum_{j=1}^n\lambda_j$, $T=\min_i X_i$, $I=\arg\min_i X_i$이면
> (a) $T\sim\mathrm{Exp}(\Lambda)$, (b) $P(I=i)=\lambda_i/\Lambda$, (c) $T$와 $I$는 독립: $P(I=i,\,T>t)=\dfrac{\lambda_i}{\Lambda}e^{-\Lambda t}$.

**가정이 왜 필요한가**
- 독립성: 없으면 $P(T>t)=\prod P(X_i>t)$로 인수분해되지 않는다 (극단적으로 $X_1=X_2$이면 $T\sim\mathrm{Exp}(\lambda_1)$이지 $\mathrm{Exp}(2\lambda_1)$이 아니다).
- 지수분포: 무기억성이 없으면 (c)가 깨진다 — 한 시계가 Gamma(2)면 '일찍 울렸다'는 정보가 '누가 울렸나'를 바꾼다 (연습문제 E3에서 확인).
- $n$ 유한 (또는 $\Lambda<\infty$): $\Lambda=\infty$면 $P(T>t)=e^{-\Lambda t}=0$, 즉 $T=0$ a.s.

<details><summary>증명</summary>

**(a)** 독립성과 생존함수로
$$P(T>t)=P(X_1>t,\dots,X_n>t)=\prod_{i=1}^n P(X_i>t)=\prod_{i=1}^n e^{-\lambda_i t}=e^{-\Lambda t},$$
즉 $T\sim\mathrm{Exp}(\Lambda)$.

**(c)를 직접 계산하면 (b)가 따라온다.** 사건 $\{I=i,\,T>t\}$는 "$X_i>t$이고 다른 모든 $X_j$가 $X_i$보다 크다"이다. $X_i=x$에 조건을 걸고 적분하면 (독립성으로 $P(X_j>x\ \forall j\ne i\mid X_i=x)=\prod_{j\ne i}e^{-\lambda_j x}$)
$$P(I=i,\,T>t)=\int_t^\infty \lambda_i e^{-\lambda_i x}\prod_{j\ne i}e^{-\lambda_j x}\,dx
=\lambda_i\int_t^\infty e^{-\Lambda x}\,dx=\frac{\lambda_i}{\Lambda}e^{-\Lambda t}.$$

**(b)** 위 식에 $t=0$을 넣으면 $P(I=i)=\lambda_i/\Lambda$. (합이 1임을 확인: $\sum_i\lambda_i/\Lambda=1$, 동률의 확률 0과 일치.)

**(c)** 결합확률이 $P(I=i,\,T>t)=\dfrac{\lambda_i}{\Lambda}\cdot e^{-\Lambda t}=P(I=i)\,P(T>t)$로 곱의 형태이므로, 모든 $i$와 $t$에서 성립하는 이 인수분해가 곧 $I\perp T$이다. $\square$

</details>

### 2.4 정리 3 — Erlang: 지수분포의 합 (Ross 12e §5.2.3, Durrett 3e §2.1)

> iid $X_i\sim\mathrm{Exp}(\lambda)$이면 $S_n=\sum_{i\le n}X_i\sim\mathrm{Gamma}(n,\lambda)$이고
> $$P(S_n>t)=\sum_{k=0}^{n-1}e^{-\lambda t}\frac{(\lambda t)^k}{k!}=P\big(\mathrm{Poisson}(\lambda t)\le n-1\big).$$

**가정이 왜 필요한가**
- 같은 비율 $\lambda$: 비율이 다르면 합은 초지수분포(hypoexponential; 밀도가 지수함수들의 선형결합)이고 Erlang이 아니다 (E1 참고).
- 독립성: 합성곱 공식 $f_{S_n}=f_{S_{n-1}}*f_X$가 독립일 때만 성립한다.

**증명 스케치.** 귀납법. $n=1$은 정의. $S_{n+1}=S_n+X_{n+1}$이고 독립이므로 합성곱
$$f_{S_{n+1}}(t)=\int_0^t f_{S_n}(s)\,\lambda e^{-\lambda(t-s)}\,ds
=\lambda e^{-\lambda t}\int_0^t\frac{\lambda^n s^{n-1}}{(n-1)!}\,ds
=\lambda e^{-\lambda t}\frac{(\lambda t)^n}{n!}.$$
꼬리 공식은 $\dfrac{d}{dt}\Big[\sum_{k<n}e^{-\lambda t}\dfrac{(\lambda t)^k}{k!}\Big]=-\lambda e^{-\lambda t}\dfrac{(\lambda t)^{n-1}}{(n-1)!}=-f_{S_n}(t)$를 확인하면 된다 (미분하면 이웃 항끼리 상쇄되는 망원합, $t=0$에서 양변 모두 1). 푸아송과의 연결 $P(S_n>t)=P(N(t)<n)$은 02b의 핵심 정리로 다시 등장한다. 완전한 증명은 Ross 12e §5.2.3 / Durrett 3e §2.1.

### 2.5 직관

지수 시계는 '늙지 않는' 시계다. 위험률이 상수 $\lambda$이므로 지금까지 얼마나 기다렸든 다음 $dt$ 안에 울릴 확률은 $\lambda\,dt$로 같다. 시계 여러 개를 동시에 돌리면 각 순간 '누군가 울릴' 비율은 $\Lambda=\sum\lambda_i$이고, 울렸을 때 그것이 $i$번일 조건부 확률은 언제나 $\lambda_i\,dt/(\Lambda\,dt)=\lambda_i/\Lambda$ — 시간에 의존하지 않으므로 '언제'와 '누가'가 독립이다. 이 그림이 그대로 CTMC(03a)의 점프 메커니즘이 된다: 상태 $x$에서 나가는 화살마다 지수 시계, 먼저 울린 화살로 점프. Erlang은 같은 시계를 $n$번 연달아 기다린 총 시간이며, $P(S_n>t)=P(N(t)<n)$은 '$n$번째 도착이 $t$ 이후'와 '$t$까지 도착이 $n$개 미만'이 같은 사건이라는 뜻이다.

### 2.6 함정(traps)

1. `rng.exponential(lam)`은 **평균이 lam**인 지수분포다. 비율 $\lambda$를 쓰려면 `rng.exponential(1/lam)`. 이 실수 하나로 모든 평균이 $\lambda^2$배 어긋난다.
2. $P(X_1<X_2)=\lambda_1/(\lambda_1+\lambda_2)$이지 $\lambda_2/(\lambda_1+\lambda_2)$가 아니다. 비율이 큰 시계가 **먼저** 울린다 (평균이 짧으므로).
3. 무기억성은 지수분포(연속)와 기하분포(이산)만의 성질이다. '평균이 같은' 균등분포나 감마분포는 기다릴수록 남은 시간 분포가 바뀐다 (4.1절의 대조군, E3).
4. $E[\min]=1/\sum\lambda_i$이지 $\min_i(1/\lambda_i)$가 아니다. 시계가 많을수록 최솟값은 어느 하나의 평균보다도 짧다.
5. 합 $S_n$의 분포는 비율이 **같을 때만** $\mathrm{Gamma}(n,\lambda)$다. 서로 다른 비율의 합은 Erlang이 아니다.

아래 빈 셀에 쓸 것:
- (1) 무기억성이 왜 지수분포에서만 성립하는지, (2) '언제 울리나'와 '누가 울리나'가 독립인 이유를 위험률 언어로, 각각 세 줄 이내로 써 보세요.
- 그 다음 정리 2의 적분 $\int_t^\infty\lambda_i e^{-\lambda_i x}\prod_{j\ne i}e^{-\lambda_j x}\,dx$를 직접 다시 계산해 보세요.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향(위/아래)이 맞는지가 목적입니다. 예측을 먼저 적어 두고 4절을 실행하세요.

In [ ]:
predictions = {
    # X ~ Exp(0.5) 일 때 P(X > 2) 는?
    "p_survive": None,
    # X ~ Exp(0.5), X > 2 인 표본만 모아 X - 2 의 평균을 내면?
    "mean_excess": None,
    # 비율 (1, 2, 3) 인 독립 지수 시계 세 개의 첫 울림 시각 T 의 평균은?
    "mean_min": None,
    # 비율 1 인 시계(가장 느린)가 첫 번째로 울릴 확률은?
    "p_clock1_wins": None,
    # '시계 1 이 이긴' 경우만 모았을 때 첫 울림 시각의 평균은? (전체 평균보다 클까, 작을까?)
    "mean_min_given_clock1": None,
    # Exp(2) 다섯 개의 합 S_5 에 대해 P(S_5 > 3) 은?
    "p_erlang_tail": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 무기억성 (정리 1)

$X\sim\mathrm{Exp}(0.5)$ (평균 2)를 뽑아 $s=2$까지 살아남은 표본의 **초과분** $X-s$를 본다. 무기억성이 맞다면 초과분은 다시 $\mathrm{Exp}(0.5)$이어야 한다. 대조군으로 같은 평균 2의 $U(0,4)$에서 같은 실험을 한다.

In [ ]:
lam, s = 0.5, 2.0
X = rng.exponential(1 / lam, N)          # scale = 1/rate = 평균 2  (함정 1)
survived = X > s
excess = X[survived] - s

U = rng.uniform(0, 4, N)                 # 대조군: 같은 평균 2, 무기억 아님
excess_U = U[U > s] - s
print(f"{N} samples; survived past s={s:g}: Exp {survived.sum()}, U(0,4) {(U > s).sum()}")

In [ ]:
x = np.linspace(0, 12, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
plot_hist_vs_pdf(excess, x, lam * np.exp(-lam * x), ax=ax1, bins=60, label_pdf="Exp(0.5) pdf")
ax1.set_xlim(0, 12); ax1.set_xlabel("excess X - s"); ax1.set_ylabel("density")
ax1.set_title("Fig 1a: Memorylessness: excess over s=2 is again Exp(0.5)")
plot_hist_vs_pdf(excess_U, x, lam * np.exp(-lam * x), ax=ax2, bins=20, label_pdf="Exp(0.5) pdf (not matching)")
ax2.plot(x, np.where(x < 2, 0.5, 0.0), "k--", label="U(0,2) pdf (actual)")
ax2.set_xlim(0, 6); ax2.set_xlabel("excess U - s"); ax2.legend()
ax2.set_title("Fig 1b: Control U(0,4): excess is U(0,2), not Exp")
plt.tight_layout(); plt.show()

In [ ]:
est_survive = mc_proportion(survived)
est_excess = mc_estimate(excess)
print(f"P(X > {s:g})        MC {est_survive}   exact e^(-lam s) = {math.exp(-lam * s):.6f}")
print(f"E[X - {s:g} | X > {s:g}]  MC {est_excess}   exact 1/lam = {1 / lam:.6f}")
print(f"control U(0,4):   E[U - 2 | U > 2] = {excess_U.mean():.4f}   (memoryless would give 2)")

$U(0,4)$는 평균이 같아도 2를 넘긴 뒤 남은 시간은 $U(0,2)$ (평균 1)로 줄어든다 — '이미 기다린 시간'이 정보가 된다. 지수분포에서만 히스토그램이 원래 밀도 위에 그대로 얹힌다.

### 4.2 경쟁하는 시계 (정리 2 a, b)

비율 $(1,2,3)$인 시계 세 개를 $N$번 동시에 돌린다. numpy는 `scale=1/rates`를 행마다 브로드캐스트하므로 `(N, 3)` 행렬 한 번에 만들어진다. 첫 울림 $T$는 행 최솟값, 승자 $I$는 `argmin` (동률 확률 0).

In [ ]:
rates = np.array([1.0, 2.0, 3.0])
Lam = rates.sum()                                    # Λ = 6
Xc = rng.exponential(1 / rates, size=(N, 3))         # 열 i 가 Exp(rates[i])
T = Xc.min(axis=1)                                   # 첫 울림 시각
I = Xc.argmin(axis=1)                                # 누가 울렸나 (0, 1, 2)
est_win = [mc_proportion(I == i) for i in range(3)]  # 승자 비율 (Fig 2b 에서 사용)
print(f"first five races: T = {T[:5].round(3)}, winner = {I[:5] + 1}")

In [ ]:
t = np.linspace(0, 1.2, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
plot_hist_vs_pdf(T, t, Lam * np.exp(-Lam * t), ax=ax1, bins=60, label_pdf="Exp(6) pdf")
ax1.set_xlim(0, 1.2); ax1.set_xlabel("T = min of three clocks"); ax1.set_ylabel("density")
ax1.set_title("Fig 2a: First ring T ~ Exp(1+2+3)")
pos = np.arange(3)
ax2.bar(pos - 0.18, [e.mean for e in est_win], 0.36, yerr=[2 * e.se for e in est_win], label="simulation (+-2 SE)")
ax2.bar(pos + 0.18, rates / Lam, 0.36, color="C1", label="lambda_i / 6")
ax2.set_xticks(pos, ["clock 1 (rate 1)", "clock 2 (rate 2)", "clock 3 (rate 3)"])
ax2.set_ylabel("P(winner = i)"); ax2.set_title("Fig 2b: Who rings first"); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
est_T = mc_estimate(T)
print(f"E[T]   MC {est_T}   exact 1/Λ = {1 / Lam:.6f}")
print(f"column means (each clock alone): {Xc.mean(0).round(3)}   exact 1/rates = {(1 / rates).round(3)}")
for i in range(3):
    print(f"P(clock {i + 1} wins)  MC {est_win[i]}   exact λ_i/Λ = {rates[i] / Lam:.6f}")

비율이 가장 큰 시계 3이 절반의 경주에서 먼저 울리고, 첫 울림의 평균 $1/6$은 가장 빠른 시계의 평균 $1/3$보다도 짧다 (함정 4).

### 4.3 '언제'와 '누가'의 독립 (정리 2 c)

독립이면 (a) 승자별 조건부 평균 $E[T\mid I=i]$가 모두 $1/\Lambda$, (b) 결합확률 $P(I=1,\,T>t_0)=\frac{\lambda_1}{\Lambda}e^{-\Lambda t_0}$, (c) 승자별 조건부 히스토그램이 모두 같은 $\mathrm{Exp}(6)$ 밀도 위에 얹힌다.

In [ ]:
fig, ax = plt.subplots()
bins = np.linspace(0, 1.2, 49)
for i in range(3):
    ax.hist(T[I == i], bins=bins, density=True, histtype="step", lw=1.5, label=f"T | winner = clock {i + 1}")
ax.plot(t, Lam * np.exp(-Lam * t), "k--", lw=2, label="Exp(6) pdf")
ax.set_xlim(0, 1.2); ax.set_xlabel("T"); ax.set_ylabel("conditional density")
ax.set_title("Fig 3: When it rings is independent of who rings"); ax.legend()
plt.show()

In [ ]:
t0 = 0.2
est_T_given = [mc_estimate(T[I == i]) for i in range(3)]
est_joint = mc_proportion((I == 0) & (T > t0))
joint_exact = rates[0] / Lam * math.exp(-Lam * t0)

for i in range(3):
    print(f"E[T | clock {i + 1} wins]  MC {est_T_given[i]}   exact 1/Λ = {1 / Lam:.6f}")
print(f"P(I=1, T>{t0})            MC {est_joint}   exact (λ_1/Λ) e^(-Λ t0) = {joint_exact:.6f}")
print(f"  product check: P(I=1) * P(T>{t0}) = {est_win[0].mean * (T > t0).mean():.5f}")

세 조건부 히스토그램이 하나로 포개진다: 시계 1(가장 느린)이 이긴 경주라고 해서 늦게 끝나지 않는다. 이것이 무기억성의 힘이다 — E3에서 시계 하나를 감마로 바꾸면 이 그림이 깨진다.

### 4.4 Erlang: 같은 시계를 $n$번 (정리 3)

$\mathrm{Exp}(2)$ 다섯 개의 합 $S_5\sim\mathrm{Gamma}(5,\ \text{rate }2)$. 평균 $n/\lambda=2.5$, 분산 $n/\lambda^2=1.25$, 꼬리 $P(S_5>3)=P(\mathrm{Poisson}(6)\le4)$.

In [ ]:
lam_e, n_e, t_e = 2.0, 5, 3.0
S = rng.exponential(1 / lam_e, size=(N, n_e)).sum(axis=1)
erlang = stats.gamma(a=n_e, scale=1 / lam_e)
print(f"first five sums S_5: {S[:5].round(3)}")

In [ ]:
u = np.linspace(0, 8, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
plot_hist_vs_pdf(S, u, erlang.pdf(u), ax=ax1, bins=60, label_pdf="Gamma(5, rate 2) pdf")
ax1.axvline(t_e, color="C3", ls="--", label="t = 3")
ax1.set_xlabel("S_5"); ax1.set_ylabel("density"); ax1.legend()
ax1.set_title("Fig 4a: Sum of five Exp(2) is Erlang(5, 2)")
tt = np.linspace(0, 5, 200)
tm = np.arange(0.25, 5, 0.5)
for n in range(1, 6):
    ax2.plot(tt, stats.gamma(a=n, scale=1 / lam_e).sf(tt), color=f"C{n - 1}", label=f"n = {n}: gamma sf")
    ax2.plot(tm, stats.poisson(lam_e * tm).cdf(n - 1), "o", color=f"C{n - 1}", ms=4)
ax2.set_xlabel("t"); ax2.set_ylabel("P(S_n > t)")
ax2.set_title("Fig 4b: lines = Gamma sf, markers = Poisson(2t).cdf(n-1)"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
est_S = mc_estimate(S)
est_tail = mc_proportion(S > t_e)
tail_exact = float(stats.poisson(lam_e * t_e).cdf(n_e - 1))
print(f"E[S_5]     MC {est_S}   exact n/λ = {n_e / lam_e:.6f}")
print(f"Var(S_5)   sample {S.var(ddof=1):.4f}   exact n/λ² = {n_e / lam_e**2:.4f}   (참고용)")
print(f"P(S_5 > 3) MC {est_tail}   exact Poisson(6).cdf(4) = {tail_exact:.6f} = gamma.sf(3) = {erlang.sf(t_e):.6f}")

오른쪽 그림의 점과 선이 겹치는 것이 정리 3의 꼬리 공식이다. 02b에서는 이 항등식을 거꾸로 읽어 — '$n$번째 도착이 $t$ 이후' $=$ '$t$까지 도착 $n$개 미만' — 도착 수 $N(t)$가 푸아송임을 얻는다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
p_survive_exact = math.exp(-lam * s)                      # e^{-λ s}
mean_excess_exact = 1 / lam                               # 무기억성
mean_min_exact = 1 / Lam                                  # Exp(Λ)
p_win_exact = rates / Lam                                 # λ_i/Λ
mean_erlang_exact = n_e / lam_e

rows = [
    {"name": "P(X > 2), X ~ Exp(0.5)", "predicted": predictions["p_survive"], "estimate": est_survive, "exact": p_survive_exact},
    {"name": "E[X − 2 ∣ X > 2]", "predicted": predictions["mean_excess"], "estimate": est_excess, "exact": mean_excess_exact},
    {"name": "E[T], T = min of clocks (1,2,3)", "predicted": predictions["mean_min"], "estimate": est_T, "exact": mean_min_exact},
    {"name": "P(clock 1 wins)", "predicted": predictions["p_clock1_wins"], "estimate": est_win[0], "exact": p_win_exact[0]},
    {"name": "P(clock 3 wins)", "predicted": None, "estimate": est_win[2], "exact": p_win_exact[2]},
    {"name": "E[T ∣ clock 1 wins]", "predicted": predictions["mean_min_given_clock1"], "estimate": est_T_given[0], "exact": mean_min_exact},
    {"name": "P(clock 1 wins, T > 0.2)", "predicted": None, "estimate": est_joint, "exact": joint_exact},
    {"name": "E[S_5], S_5 = sum of five Exp(2)", "predicted": None, "estimate": est_S, "exact": mean_erlang_exact},
    {"name": "P(S_5 > 3)", "predicted": predictions["p_erlang_tail"], "estimate": est_tail, "exact": tail_exact},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **P(X > 2)**: 어긋나면 `rng.exponential`의 scale 규약(평균 $1/\lambda$)을 잘못 쓴 것이다 (함정 1).
- **E[X − 2 | X > 2]**: 어긋나면 무기억성이 깨진 것 — 지수분포가 아닌 표본을 뽑았다는 뜻.
- **E[T]**: 어긋나면 최솟값의 비율이 비율의 합 $\Lambda$라는 정리 2(a)를 의심한다 (또는 `min(axis=1)`의 축 실수).
- **P(clock 1 wins), P(clock 3 wins)**: 어긋나면 승자 확률 $\lambda_i/\Lambda$의 방향(느린 시계가 이길 확률이 작다)을 확인한다 (함정 2).
- **E[T | clock 1 wins]**: 어긋나면 '언제'와 '누가'의 독립(정리 2c)이 깨진 것 — 지수가 아닌 시계가 섞였다는 뜻.
- **P(clock 1 wins, T > 0.2)**: 결합확률이 곱으로 인수분해되는지의 직접 검사. 어긋나면 위와 같은 원인.
- **E[S_5]**: 어긋나면 `sum(axis=1)`의 축 또는 scale 규약 문제다.
- **P(S_5 > 3)**: 어긋나면 Erlang 꼬리와 푸아송 cdf의 항등식(정리 3)에서 $n-1$을 $n$으로 잘못 쓴 것일 가능성이 크다.

In [ ]:
assert_close(est_survive, p_survive_exact, k=4, name="p_survive")
assert_close(est_excess, mean_excess_exact, k=4, name="mean_excess")
assert_close(est_T, mean_min_exact, k=4, name="mean_min")
assert_close(est_win[0], p_win_exact[0], k=4, name="p_clock1_wins")
assert_close(est_win[2], p_win_exact[2], k=4, name="p_clock3_wins")
assert_close(est_T_given[0], mean_min_exact, k=4, name="mean_min_given_clock1")
assert_close(est_joint, joint_exact, k=4, name="p_clock1_and_min_gt_t0")
assert_close(est_S, mean_erlang_exact, k=4, name="mean_erlang")
assert_close(est_tail, tail_exact, k=4, name="p_erlang_tail")
print("all checks passed")

## 6. 연습문제

### E1 계산

비율 $(1,2,3)$인 독립 지수 시계 세 개.
(a) 시계 2가 첫 번째, 시계 3이 두 번째로 울릴 확률은?
(b) 첫 울림과 두 번째 울림 사이 시간의 기댓값은?
(c) 세 시계가 모두 울릴 때까지 걸리는 시간(최댓값)의 기댓값은?
손으로 계산하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 첫 승자가 시계 2일 확률은 $2/6$. 시계 2가 울린 순간 무기억성으로 남은 두 시계는 **새로 시작한** $\mathrm{Exp}(1)$, $\mathrm{Exp}(3)$과 같으므로 시계 3이 다음일 확률은 $3/(1+3)=3/4$. 곱: $\dfrac26\cdot\dfrac34=\dfrac14$.

(b) 순차 논법. 첫 간격은 $\mathrm{Exp}(6)$. 첫 승자가 $i$이면 남은 시계들의 비율 합은 $6-\lambda_i$이고, 무기억성으로 두 번째 간격은 $\mathrm{Exp}(6-\lambda_i)$이다. 전확률:
$$E[\text{2번째 간격}]=\frac16\cdot\frac15+\frac26\cdot\frac14+\frac36\cdot\frac13=\frac1{30}+\frac1{12}+\frac16=\frac{17}{60}\approx0.28333.$$

(c) 포함-배제. $P(\max>x)=1-\prod_i(1-e^{-\lambda_i x})=\sum_i e^{-\lambda_i x}-\sum_{i<j}e^{-(\lambda_i+\lambda_j)x}+e^{-\Lambda x}$이고 $E[\max]=\int_0^\infty P(\max>x)\,dx$이므로
$$E[\max]=\sum_i\frac1{\lambda_i}-\sum_{i<j}\frac1{\lambda_i+\lambda_j}+\frac1{\Lambda}
=\Big(1+\frac12+\frac13\Big)-\Big(\frac13+\frac14+\frac15\Big)+\frac16=\frac{73}{60}\approx1.21667.$$
검산: 세 간격의 합이 최댓값이므로 $E[\max]=\frac16+\frac{17}{60}+E[\text{3번째 간격}]$, 즉 $E[\text{3번째 간격}]=\frac{73-10-17}{60}=\frac{46}{60}$ — 마지막 남은 시계가 느린 시계일 확률이 높아 마지막 간격이 가장 길다.

검산 코드 (분수 산술):
```python
from fractions import Fraction as F
r = [F(1), F(2), F(3)]
gap2 = sum(ri / 6 * 1 / (6 - ri) for ri in r)
emax = sum(1 / ri for ri in r) - sum(1 / (r[i] + r[j]) for i in range(3) for j in range(i + 1, 3)) + F(1, 6)
print(F(2, 6) * F(3, 4), gap2, emax)     # 1/4  17/60  73/60
```

</details>

### E2 유도 후 시뮬레이션 검증

$E[\max(X_1,X_2,X_3)]=73/60$ (비율 1, 2, 3)을 포함-배제로 유도한 뒤 시뮬레이션으로 검증하라. 추가로 E1(b)의 순차 논법(첫 승자에 조건부로 남은 시계들의 비율 합이 줄어든다)을 코드로 재현해 두 번째 간격의 평균 $17/60$도 확인하라. 힌트: 행마다 정렬한 열의 차이가 곧 간격이다.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

유도는 E1(c)와 같다: $P(\max>x)=1-\prod_i(1-e^{-\lambda_i x})$를 전개해 항별로 $\int_0^\infty e^{-cx}dx=1/c$를 적용한다.

```python
Xe = rng.exponential(1 / rates, (N, 3))
Xs = np.sort(Xe, axis=1)                          # 열 0,1,2 = 첫/둘째/셋째 울림 시각
est_max = mc_estimate(Xs[:, 2])                   # = Xe.max(1)
est_gap2 = mc_estimate(Xs[:, 1] - Xs[:, 0])
print(est_max, " exact 73/60 =", 73 / 60)          # 1.2167 ± 0.002 (full)
print(est_gap2, " exact 17/60 =", 17 / 60)         # 0.2833 ± 0.0006 (full)
assert_close(est_max, 73 / 60, k=4, name="E[max]")
assert_close(est_gap2, 17 / 60, k=4, name="E[second gap]")

# 순차 논법: 첫 승자 i 에 조건부로 두 번째 간격 ~ Exp(6 - λ_i)
I1 = Xe.argmin(axis=1)
for i in range(3):
    g = Xs[I1 == i, 1] - Xs[I1 == i, 0]
    print(f"winner {i + 1}: E[gap2 | winner] = {g.mean():.4f}   exact 1/(6 - λ_i) = {1 / (6 - rates[i]):.4f}")
```

세 조건부 평균 $1/5, 1/4, 1/3$을 승자 확률 $1/6, 2/6, 3/6$으로 가중평균하면 $17/60$이 나온다.

</details>

### E3 가정을 깨보기

시계 1을 지수분포 대신 **평균이 같은** $\mathrm{Gamma}(2,\ \text{rate}=2)$ (평균 1)로 바꾸고 시계 2는 $\mathrm{Exp}(1)$로 둔다.
(a) $P(\text{시계 1이 먼저})$가 여전히 $1/2$인가?
(b) 승자와 최솟값 시각이 독립인가 — $E[T\mid I=1]$과 $E[T\mid I=2]$를 비교하라.
(c) 무기억성: $U(0,2)$ (평균 1) 시계에 대해 $E[X-1\mid X>1]$을 시뮬레이션하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $Y\sim\mathrm{Gamma}(2,\text{rate }2)$, $Z\sim\mathrm{Exp}(1)$ 독립. $P(Y<Z)=E[P(Z>Y\mid Y)]=E[e^{-Y}]=M_Y(-1)=\Big(\dfrac{2}{2+1}\Big)^2=\dfrac49\ne\dfrac12$. 감마 시계는 위험률이 0에서 시작해 초반에 잘 안 울린다.

(b) 독립성이 깨진다. $E[Y;\,Y<Z]=E[Ye^{-Y}]=\int_0^\infty y\cdot4ye^{-2y}e^{-y}dy=4\cdot\dfrac{2!}{3^3}=\dfrac{8}{27}$이므로 $E[T\mid I=1]=\dfrac{8/27}{4/9}=\dfrac23$; $E[Z;\,Z<Y]=\int_0^\infty ze^{-z}e^{-2z}(1+2z)\,dz=\dfrac19+\dfrac4{27}=\dfrac7{27}$이므로 $E[T\mid I=2]=\dfrac{7/27}{5/9}=\dfrac7{15}$. 시계 1이 이긴 경주는 늦게 끝난 쪽으로 치우친다: $2/3>7/15$, 두 조건부 평균이 4 SE 훨씬 밖에서 갈라진다.

(c) $X-1\mid X>1\sim U(0,1)$, 평균 $0.5\ne1$. 무기억성이 없으면 '이미 기다린 시간'이 정보가 된다.

```python
Y = rng.gamma(2.0, 1 / 2, N)                      # Gamma(shape 2, scale 1/2): rate 2, 평균 1
Z = rng.exponential(1.0, N)                       # Exp(1)
win1 = Y < Z
T3 = np.minimum(Y, Z)
print(mc_proportion(win1), " exact 4/9 =", 4 / 9)                       # 0.444 ± 0.004 (FAST)
print(mc_estimate(T3[win1]), " exact 2/3;", mc_estimate(T3[~win1]), " exact 7/15 =", 7 / 15)
assert_close(mc_proportion(win1), 4 / 9, k=4, name="P(gamma clock first)")
assert_close(mc_estimate(T3[win1]), 2 / 3, k=4, name="E[T | I=1]")
assert_close(mc_estimate(T3[~win1]), 7 / 15, k=4, name="E[T | I=2]")
U2 = rng.uniform(0, 2, N)
print(mc_estimate(U2[U2 > 1] - 1), " exact 0.5 (memoryless would give 1)")
```

</details>

## 7. 정리

1. $\mathrm{Exp}(\lambda)$는 무기억 $P(X>s+t\mid X>s)=P(X>t)$이고, 연속분포 중 이 성질을 가진 것은 지수분포뿐이다 (함수방정식 $g(s+t)=g(s)g(t)$의 단조 해는 $e^{-\lambda t}$뿐).
2. 독립 지수 시계 $n$개: 첫 울림 $T\sim\mathrm{Exp}(\sum\lambda_i)$, 승자 $P(I=i)=\lambda_i/\sum\lambda_j$, 그리고 $T\perp I$ — 한 번의 적분 $\int_t^\infty\lambda_i e^{-\Lambda x}dx=\frac{\lambda_i}{\Lambda}e^{-\Lambda t}$로 세 가지가 동시에 나온다.
3. '언제'와 '누가'의 독립은 위험률이 상수이기 때문이며, 이것이 CTMC의 점프 규칙(03a)과 푸아송 과정의 중첩(02c)의 뿌리다. 시계 하나만 감마로 바꿔도 깨진다 (E3).
4. 같은 비율 지수 $n$개의 합은 $\mathrm{Gamma}(n,\lambda)$이고 $P(S_n>t)=P(\mathrm{Poisson}(\lambda t)\le n-1)$ — 다음 노트북의 푸아송 과정 정의로 직행하는 항등식.
5. 코드 규약: numpy `rng.exponential(scale)`의 scale은 평균 $1/\lambda$; scipy `stats.gamma(a=n, scale=1/λ)`.

**trap 카드**
- Q: 비율 1, 2, 3인 시계 중 '비율 3'이 가장 늦게 울릴 것 같지 않은가? 그리고 세 시계의 첫 울림 평균은 가장 빠른 시계의 평균 $1/3$보다 길까 짧을까?
- A: 비율이 크면 평균 대기시간 $1/\lambda$이 짧으므로 비율 3이 **가장 자주 먼저** 울린다 (확률 1/2). 첫 울림의 평균은 $1/(1+2+3)=1/6$로 어느 시계 하나의 평균보다 짧다 — 최솟값의 비율은 비율들의 합이다.

**다음 노트북: 02b** — iid $\mathrm{Exp}(\lambda)$ 간격을 이어 붙이면 도착시각 $T_n\sim\mathrm{Gamma}(n,\lambda)$이 되고, $P(T_n\le t)=P(N(t)\ge n)$이 $N(t)\sim\mathrm{Poisson}(\lambda t)$를 준다. 02b는 이 관찰을 푸아송 과정의 세 가지 동치 정의로 확장하고, 도착시각이 균등 순서통계량임을 보인다.

log/progress.md 한 줄 템플릿:
`- YYYY-MM-DD 02a 무기억성과 경쟁하는 시계 | 소요 __분 | 예측 적중 _/6 | 막힌 곳: ___ | 다음: 02b`